In [1]:
# ============================================================
# PACKAGE 6
# COMPLETE EDA + SATELLITE FEATURE IMPORTANCE
# AI Urban Air Quality Intelligence
# ============================================================


import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

import seaborn as sns



# ============================================================
# PATHS
# ============================================================


MASTER_FILE = r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Data_Integration\Delhi_Master_Dataset_Final.csv"


SATELLITE_FILE = (
r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE"
r"\Feature_Engineering\Satellite0"
r"\Delhi_Satellite_Feature_Engineered.csv"
)



OUTPUT = "EDA_Report"



PLOT_DIR = os.path.join(
OUTPUT,
"Plots"
)


REPORT_DIR = os.path.join(
OUTPUT,
"Reports"
)


os.makedirs(
PLOT_DIR,
exist_ok=True
)


os.makedirs(
REPORT_DIR,
exist_ok=True
)



# ============================================================
# LOAD DATA
# ============================================================


print("\nLoading datasets...")


df = pd.read_csv(
MASTER_FILE
)


satellite = pd.read_csv(
SATELLITE_FILE
)



df["Date"] = pd.to_datetime(
df["Date"]
)


satellite["Date"] = pd.to_datetime(
satellite["Date"]
)



df = df.sort_values(
"Date"
)



print(
"Master Dataset:",
df.shape
)


print(
"Satellite Dataset:",
satellite.shape
)




# ============================================================
# DATASET HEALTH CHECK
# ============================================================


print("\n==========================")
print("DATASET INFORMATION")
print("==========================")


print(df.info())


print("\nMissing Values:")

print(

df.isnull()
.sum()
.sort_values(
ascending=False
)
.head(20)

)



print(
"\nDuplicate Rows:",
df.duplicated().sum()
)





# ============================================================
# AQI DISTRIBUTION
# ============================================================


plt.figure(figsize=(8,5))

plt.hist(
df["AQI"],
bins=50
)

plt.xlabel("AQI")

plt.ylabel("Frequency")

plt.title(
"AQI Distribution"
)


plt.savefig(
os.path.join(
PLOT_DIR,
"AQI_Distribution.png"
),
bbox_inches="tight"
)


plt.close()





# ============================================================
# AQI TIME TREND
# ============================================================


plt.figure(figsize=(12,5))


plt.plot(

df["Date"],

df["AQI"]

)


plt.xlabel("Date")

plt.ylabel("AQI")

plt.title(
"Delhi AQI Trend"
)


plt.savefig(

os.path.join(
PLOT_DIR,
"AQI_Time_Trend.png"
),

bbox_inches="tight"

)


plt.close()





# ============================================================
# MONTHLY AQI
# ============================================================


df["Month_Num"] = df["Date"].dt.month



monthly_aqi = (

df.groupby(
"Month_Num"
)["AQI"]
.mean()

)



plt.figure(figsize=(8,5))


plt.plot(

monthly_aqi.index,

monthly_aqi.values,

marker="o"

)


plt.xlabel(
"Month"
)


plt.ylabel(
"Average AQI"
)


plt.title(
"Monthly AQI Pattern"
)



plt.savefig(

os.path.join(
PLOT_DIR,
"Monthly_AQI.png"
),

bbox_inches="tight"

)


plt.close()





# ============================================================
# SEASONAL AQI
# ============================================================


def season(month):

    if month in [12,1,2]:
        return "Winter"

    elif month in [3,4,5]:
        return "Summer"

    elif month in [6,7,8,9]:
        return "Monsoon"

    else:
        return "Post-Monsoon"



df["Season_EDA"] = (

df["Date"]
.dt.month
.apply(season)

)



season_aqi = (

df.groupby(
"Season_EDA"
)["AQI"]
.mean()

)



plt.figure(figsize=(8,5))


plt.bar(

season_aqi.index,

season_aqi.values

)


plt.xlabel(
"Season"
)


plt.ylabel(
"Average AQI"
)


plt.title(
"Seasonal AQI Variation"
)



plt.savefig(

os.path.join(
PLOT_DIR,
"Seasonal_AQI.png"
),

bbox_inches="tight"

)


plt.close()





# ============================================================
# CORRELATION FUNCTION
# ============================================================


def correlation_heatmap(
data,
filename,
title
):


    corr = data.corr()



    plt.figure(figsize=(12,8))


    sns.heatmap(

        corr,

        cmap="coolwarm",

        center=0

    )


    plt.title(title)


    plt.savefig(

    os.path.join(
    PLOT_DIR,
    filename
    ),

    bbox_inches="tight"

    )


    plt.close()





# ============================================================
# POLLUTANT CORRELATION
# ============================================================


pollutants = [

"PM2.5",
"PM10",
"NO",
"NO2",
"NOx",
"NH3",
"CO",
"SO2"

]


pollutants = [

c for c in pollutants

if c in df.columns

]



pollution_df = df[

pollutants+
["AQI"]

]



pollution_corr = (

pollution_df
.corr()["AQI"]
.sort_values(
ascending=False
)

)



pollution_corr.to_csv(

os.path.join(
REPORT_DIR,
"AQI_Correlation.csv"
)

)



correlation_heatmap(

pollution_df,

"Pollutant_Correlation_Heatmap.png",

"Pollutant Correlation"

)





# ============================================================
# WEATHER CORRELATION
# ============================================================


weather_cols = [

c for c in df.columns

if any(

x in c.lower()

for x in [

"temp",
"humidity",
"wind",
"pressure",
"rain",
"precip"

]

)

]



weather_df = df[

weather_cols+

["AQI"]

]



weather_df = weather_df.select_dtypes(

include=np.number

)



weather_corr = (

weather_df
.corr()["AQI"]
.sort_values(
ascending=False
)

)



weather_corr.to_csv(

os.path.join(
REPORT_DIR,
"Weather_Correlation.csv"
)

)



if len(weather_df.columns)>1:

    correlation_heatmap(

    weather_df,

    "Weather_Correlation_Heatmap.png",

    "Weather Correlation"

    )





# ============================================================
# SATELLITE CORRELATION
# ============================================================


sat_df = pd.merge(

df[

[
"Date",
"AQI"
]

],

satellite,

on="Date",

how="inner"

)



sat_numeric = sat_df.select_dtypes(

include=np.number

)



sat_corr = (

sat_numeric
.corr()["AQI"]
.sort_values(
ascending=False
)

)



sat_corr.to_csv(

os.path.join(
REPORT_DIR,
"Satellite_Correlation.csv"
)

)



correlation_heatmap(

sat_numeric,

"Satellite_Correlation_Heatmap.png",

"Satellite Feature Correlation"

)





# ============================================================
# COMPLETE FEATURE CORRELATION
# ============================================================


numeric_df = df.select_dtypes(

include=np.number

)



correlation_heatmap(

numeric_df,

"Feature_Correlation_Heatmap.png",

"Complete Feature Correlation"

)





# ============================================================
# RANDOM FOREST FEATURE IMPORTANCE
# ============================================================


print(
"\nTraining Random Forest for importance..."
)



model_data = df.copy()



X = model_data.drop(

columns=[

"Date",
"AQI"

],

errors="ignore"

)



y = model_data["AQI"]





for col in X.select_dtypes(

include="object"

).columns:


    encoder = LabelEncoder()


    X[col] = encoder.fit_transform(

    X[col].astype(str)

    )



X = X.fillna(
X.median()
)



rf = RandomForestRegressor(

n_estimators=200,

random_state=42,

n_jobs=-1

)



rf.fit(

X,

y

)




importance = pd.DataFrame({

"Feature":X.columns,

"Importance":rf.feature_importances_

})



importance = importance.sort_values(

"Importance",

ascending=False

)



importance.to_csv(

os.path.join(
REPORT_DIR,
"Feature_Importance.csv"
),

index=False

)





# ============================================================
# SATELLITE FEATURE IMPORTANCE
# ============================================================


satellite_importance = importance[

importance["Feature"]

.str.contains(

"MODIS|Sentinel",

regex=True

)

]



satellite_importance.to_csv(

os.path.join(

REPORT_DIR,

"Satellite_Feature_Importance.csv"

),

index=False

)



plt.figure(figsize=(10,6))


plt.barh(

satellite_importance["Feature"],

satellite_importance["Importance"]

)


plt.xlabel(
"Importance"
)


plt.title(
"Satellite Feature Importance"
)



plt.gca().invert_yaxis()



plt.savefig(

os.path.join(

PLOT_DIR,

"Satellite_Feature_Importance.png"

),

bbox_inches="tight"

)



plt.close()





# ============================================================
# AQI HOTSPOT DAYS
# ============================================================


hotspots = (

df.sort_values(

"AQI",

ascending=False

)

.head(50)

)



hotspots[

[
"Date",
"AQI"

]

].to_csv(

os.path.join(

REPORT_DIR,

"Top_Pollution_Days.csv"

),

index=False

)



plt.figure(figsize=(10,5))


plt.plot(

hotspots["Date"],

hotspots["AQI"],

marker="o"

)


plt.xticks(rotation=45)


plt.xlabel(
"Date"
)


plt.ylabel(
"AQI"
)


plt.title(
"Top AQI Hotspot Days"
)



plt.savefig(

os.path.join(

PLOT_DIR,

"AQI_Hotspot_Days.png"

),

bbox_inches="tight"

)


plt.close()




print(
"""
================================
PACKAGE 6 COMPLETED SUCCESSFULLY
================================

Generated:
✔ AQI EDA
✔ Weather EDA
✔ Satellite EDA
✔ Correlation Reports
✔ Feature Importance
✔ Satellite Importance
✔ Pollution Hotspot Analysis

Next:
Package 7 - Feature Selection + Leakage Check
"""
)


Loading datasets...
Master Dataset: (2009, 240)
Satellite Dataset: (1714, 14)

DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Columns: 240 entries, City to Unnamed_Facility_Count
dtypes: datetime64[ns](1), float64(129), int64(107), object(3)
memory usage: 3.7+ MB
None

Missing Values:
Sentinel_AAI_Lag3    1360
Sentinel_NO2_Lag3    1360
Sentinel_NO2_Lag1    1358
Sentinel_AAI_Lag1    1358
Sentinel_NO2         1357
Sentinel_AAI_MA3     1357
Sentinel_AAI         1357
Sentinel_NO2_MA3     1357
MODIS_AOD_Lag3       1118
MODIS_AOD_Lag1       1116
MODIS_AOD_MA7        1115
MODIS_AOD            1115
MODIS_AOD_MA3        1115
City                    0
PM2.5                   0
Date                    0
Year                    0
Month                   0
Day                     0
DayOfWeek               0
dtype: int64

Duplicate Rows: 0

Training Random Forest for importance...

PACKAGE 6 COMPLETED SUCCESSFULLY

Generated:
✔ AQI EDA
✔ Weather EDA
✔ 